# Load dependencies

first run this command `uv add datasets` in the terminal

In [ ]:
from datasets import load_dataset
from pprint import pprint
import html

# Loading a local dataset

In [ ]:
# !wget -P data https://github.com/crux82/squad-it/raw/master/SQuAD_it-train.json.gz
# !wget -P data https://github.com/crux82/squad-it/raw/master/SQuAD_it-test.json.gz
# !gzip -dkv data/SQuAD_it-*.json.gz

- To load a JSON file with the `load_dataset()` function, we just need to know if we’re dealing with ordinary **JSON** (similar to a nested dictionary) or **JSON Lines** (line-separated JSON).
- SQuAD-it uses the nested format, with all the text stored in a `data` field.
- This means we can load the dataset by specifying the `field` argument as follows:

## A single file path --> defaults to `train` split

In [ ]:
squad_it_dataset = load_dataset("json", data_files="data/SQuAD_it-train.json", field="data")
print(squad_it_dataset)

sample = squad_it_dataset["train"][0]
print(sample.keys())
print(sample['title'])
print(len(sample['paragraphs']))
pprint(sample['paragraphs'][0])

In [ ]:
# this would also put all data parts in the train split.
squad_it_dataset = load_dataset("json", data_files="data/*.json", field="data")
print(squad_it_dataset)

## A dict of file paths

In [ ]:
data_files = {"train": "data/SQuAD_it-train.json", "test": "data/SQuAD_it-test.json"}
squad_it_dataset = load_dataset("json", data_files=data_files, field="data")
print(squad_it_dataset)

the `load_dataset` also handles the unzipping automatically

In [ ]:
data_files = {"train": "data/SQuAD_it-train.json.gz", "test": "data/SQuAD_it-test.json.gz"}
squad_it_dataset = load_dataset("json", data_files=data_files, field="data")
print(squad_it_dataset)

# Loading a remote dataset

In [ ]:
url = "https://github.com/crux82/squad-it/raw/master/"
data_files = {
    "train": url + "SQuAD_it-train.json.gz",
    "test": url + "SQuAD_it-test.json.gz",
}
squad_it_dataset = load_dataset("json", data_files=data_files, field="data")
print(squad_it_dataset)

# Data preprocessing (slice and dice)

In [ ]:
# data_path = "https://archive.ics.uci.edu/ml/machine-learning-databases/00462/drugsCom_raw.zip"
# !wget -P data {data_path}
# !unzip -o data/drugsCom_raw.zip -d data/

In [ ]:
data_files = {"train": "data/drugsComTrain_raw.tsv", "test": "data/drugsComTest_raw.tsv"}
drug_dataset = load_dataset("csv", data_files=data_files, delimiter="\t")
print(drug_dataset)

## Sampling for initial review

In [ ]:
drug_sample = drug_dataset["train"]\
    .shuffle(seed=42)\
    .select(range(1000)) # expects an iterable of indices
drug_sample[:3]

- The `Unnamed: 0` column looks suspiciously like an anonymized ID for each patient.
- The `condition` column includes a mix of uppercase and lowercase labels.
- The `reviews` are of varying length and contain a mix of Python line separators (`\r\n`) as well as HTML character codes like `&\#039;`.

## Addressing each issue with a proper preprocessing solution

1. Verify that the number of IDs matches the number of rows in each split.
- if it's verified, rename the ID column to a proper name

In [ ]:
print(drug_dataset.keys())
for split in drug_dataset.keys():
    assert len(drug_dataset[split]) == len(drug_dataset[split].unique("Unnamed: 0"))

# # find the number of unique drugs and conditions in the training and test sets.
# for split in drug_dataset.keys():
#     print(f"split: {split}")
#     for col in ["condition", "drugName"]:
#         unique_count = drug_dataset[split].unique(col).__sizeof__()
#         print(f"number of {col} unique values: {unique_count}")

In [ ]:
drug_dataset = drug_dataset.rename_column(
    original_column_name="Unnamed: 0", new_column_name="patient_id"
)
print(drug_dataset)

2. Next, let’s normalize all the `condition` labels using `Dataset.map()`.

In [ ]:
def lowercase_condition(example):
    return {"condition": example["condition"].lower()}
# drug_dataset.map(lowercase_condition)
# AttributeError: 'NoneType' object has no attribute 'lower'
drug_dataset = drug_dataset.filter(lambda x: x["condition"] is not None)
drug_dataset = drug_dataset.map(lowercase_condition)

In [ ]:
(drug_dataset["train"]["condition"][:3])

3. Handling `review` length distribution
- Whenever you’re dealing with customer reviews, a good practice is to check the number of words in each review.
- A review might be just a single word like “Great!” or a full-blown essay with thousands of words
- and depending on the use case you’ll need to handle these extremes differently.

An alternative way to add new columns to a dataset is with the `Dataset.add_column()` function.

This allows you to provide the column as a Python **list or NumPy array** and can be handy in situations where `Dataset.map()` is not well suited for your analysis.

In [ ]:
def compute_review_length(example):
    return {"review_length": len(example["review"].split())}
drug_dataset = drug_dataset.map(compute_review_length)
pprint(drug_dataset["train"][0])

In [ ]:
import matplotlib.pyplot as plt
fig, axs = plt.subplots(1,2, sharey=True, figsize = (10,5))
axs[0].hist(drug_dataset['train']['review_length'], bins=50, log=True);
drug_dataset = drug_dataset.filter(lambda x: x["review_length"] > 30)
print(drug_dataset.num_rows)
axs[1].hist(drug_dataset['train']['review_length'], bins=50, log=True);

4. Hnadle HTML characters in reviews.
- The last thing we need to deal with is the presence of HTML character codes in our reviews.
- We can use Python’s `html` module to unescape these characters.

In [ ]:
text = "I&#039;m a transformer called BERT"
html.unescape(text)

In [ ]:
drug_dataset = drug_dataset.map(lambda x: {"review": html.unescape(x["review"])})

## Batched `Dataset` ops
- If set to `True`, causes it to send a batch of examples to the map function at once
- The `batch_size` is configurable but defaults to 1,000.
- The received batch has the fields of the dataset, but each value is now a list of values, and not just a single value.
- The return value of `Dataset.map()` should be the same:
    - a dictionary with the fields we want to update or add to our dataset
    - and a list of values

### Pipeline time without batch

In [ ]:
# !rm -rf ~/.cache/huggingface/datasets/

In [ ]:
# data_files = {"train": "data/drugsComTrain_raw.tsv", "test": "data/drugsComTest_raw.tsv"}

# def lowercase_condition(example):
#     return {"condition": example["condition"].lower()}

# def compute_review_length(example):
#     return {"review_length": len(example["review"].split())}

# drug_dataset = load_dataset("csv", data_files=data_files, delimiter="\t")
# drug_dataset = drug_dataset.rename_column(
#     original_column_name="Unnamed: 0", new_column_name="patient_id"
# )

In [ ]:
# %%time
# drug_dataset = drug_dataset.filter(lambda x: x["condition"] is not None)
# drug_dataset = drug_dataset.map(lowercase_condition)
# drug_dataset = drug_dataset.map(compute_review_length)
# drug_dataset = drug_dataset.filter(lambda x: x["review_length"] > 30)
# drug_dataset = drug_dataset.map(lambda x: {"review": html.unescape(x["review"])})

### Pipeline time with batch

In [ ]:
!rm -rf ~/.cache/huggingface/datasets/

In [ ]:
def lowercase_condition(batch):
    return {
        "condition": [c.lower() for c in batch["condition"]]
    }

def compute_review_length(batch):
    return {
        "review_length": [len(r.split()) for r in batch["review"]]
    }

drug_dataset = load_dataset("csv", data_files=data_files, delimiter="\t")
drug_dataset = drug_dataset.rename_column("Unnamed: 0", "patient_id")

In [ ]:
%%time
drug_dataset = drug_dataset.filter(
    lambda batch: [c is not None for c in batch["condition"]],
    batched=True, batch_size = 1000,
)
drug_dataset = drug_dataset.map(lowercase_condition, batched=True, batch_size = 1000)
drug_dataset = drug_dataset.map(compute_review_length, batched=True, batch_size = 1000)
drug_dataset = drug_dataset.filter(
    lambda batch: [rl > 30 for rl in batch["review_length"]],
    batched=True, batch_size = 1000,
)
drug_dataset = drug_dataset.map(
    lambda batch: {"review": [html.unescape(r) for r in batch["review"]]},
    batched=True, batch_size = 1000,
)

### Tokenization w/wo batch

Using `Dataset.map()` with `batched=True` will be essential to unlock the speed of the **“fast” tokenizers**

In [ ]:
from transformers import AutoTokenizer
fast_tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")
slow_tokenizer = AutoTokenizer.from_pretrained("bert-base-cased", use_fast=False)
def tokenize_function(examples, tokenizer):
    return tokenizer(examples["review"], truncation=True)
    # the tokenizer itself handle the list or iterable input values.

In [ ]:
# %time tokenized_dataset = drug_dataset.map(tokenize_function, batched=False, fn_kwargs={"tokenizer": fast_tokenizer})

In [ ]:
# %time tokenized_dataset = drug_dataset.map(tokenize_function, batched=True, fn_kwargs={"tokenizer": fast_tokenizer})

In [ ]:
# %time tokenized_dataset = drug_dataset.map(tokenize_function, batched=True, num_proc=8, fn_kwargs={"tokenizer": fast_tokenizer})

In [ ]:
# %time tokenized_dataset = drug_dataset.map(tokenize_function, batched=False, fn_kwargs={"tokenizer": slow_tokenizer})

In [ ]:
# %time tokenized_dataset = drug_dataset.map(tokenize_function, batched=True, fn_kwargs={"tokenizer": slow_tokenizer})

In [ ]:
# %time tokenized_dataset = drug_dataset.map(tokenize_function, batched=True, num_proc=8, fn_kwargs={"tokenizer": slow_tokenizer})

- In general, it's not recommended to use Python multiprocessing for fast tokenizers with `batched=True`.
- Using `num_proc` to speed up your processing is usually a great idea, as long as the function you are using is not already doing some kind of multiprocessing of its own.

## Creating multiple recrods from a single record
- With `Dataset.map()` and `batched=True` you can change the number of elements in your dataset.
- This is super useful in many situations (like here and for question answering) where you want to create several training features from one example

- In the following example, we will tokenize our examples and truncate them to a maximum length of 128
- But we will ask the tokenizer to return all the chunks of the texts instead of just the first one.
- This can be done with `return_overflowing_tokens=True`:

In [ ]:
def tokenize_and_split(examples):
    return fast_tokenizer(
        examples["review"],
        truncation=True,
        max_length=128,
        return_overflowing_tokens=True,
    )

result = tokenize_and_split(drug_dataset["train"][0])
[len(inp) for inp in result["input_ids"]]

In [ ]:
# tokenized_dataset = drug_dataset.map(tokenize_and_split, batched=True)
# # ArrowInvalid: Column 8 named input_ids expected length 1000 but got length 1463

- The problem is that we’re trying to mix two different datasets of different sizes:
    - The `drug_dataset` columns will have a certain number of examples (the 1,000 in our error)
    - But the `tokenized_dataset` we are building will have more (the 1,463 in the error message)
        - it is more than 1,000 because we are tokenizing long reviews into more than one example by using `return_overflowing_tokens=True`.

The solution:
- Remove the columns from the old dataset
    - with the `remove_columns` argument
- Make them the same size as they are in the new dataset.
    - Using `overflow_to_sample_mapping`

### First Option

In [ ]:
tokenized_dataset = drug_dataset.map(
    tokenize_and_split, batched=True, remove_columns=drug_dataset["train"].column_names
)

In [ ]:
print(tokenized_dataset["train"].features)
print(len(tokenized_dataset["train"]), len(drug_dataset["train"]))

### Second Option

In [ ]:
def tokenize_and_split(examples):
    result = fast_tokenizer(
        examples["review"],
        truncation=True,
        max_length=128,
        return_overflowing_tokens=True,
    )
    # Extract mapping between new and old indices
    sample_map = result.pop("overflow_to_sample_mapping")
    for key, values in examples.items():
        result[key] = [values[i] for i in sample_map]
    return result

tokenized_dataset = drug_dataset.map(tokenize_and_split, batched=True)
tokenized_dataset

## From Datasets to DataFrames and back

- To enable the conversion between various third-party libraries, Datasets provides a `Dataset.set_format()` function.
- This function **only** changes the **output format** of the dataset
- So, you can easily switch to another format **without affecting** the underlying data format, which is **Apache Arrow**.
- The formatting is done in place.

In [ ]:
drug_dataset.set_format("pandas")
drug_dataset["train"][:3]

Creating a `pandas.DataFrame` for the whole training set by selecting all the elements of `drug_dataset["train"]`
- Under the hood, `Dataset.set_format()` changes the return format for the dataset’s `__getitem__()` dunder method.
- This means that when we want to create a new object like `train_df` from a Dataset in the "pandas" format
    - we need to slice the whole dataset to obtain a `pandas.DataFrame`.
- You can verify for yourself that the type of `drug_dataset["train"]` is `Dataset`, irrespective of the output format.

In [ ]:
train_df = drug_dataset["train"][:]
train_df.head()

In [ ]:
frequencies = (
    train_df["condition"]
    .value_counts()
    .to_frame()
    .reset_index()
    .rename(columns={"index": "condition", "count": "frequency"})
)

avg_rating_per_drug = train_df\
    .groupby("drugName")\
    .agg(avg_rating = ('rating', "mean"))\
    .reset_index()

display(frequencies.head())
display(avg_rating_per_drug.head())

Once we’re done with our Pandas analysis, we can always create a new `Dataset` object by using the `Dataset.from_pandas()` function as follows:

In [ ]:
from datasets import Dataset
freq_dataset = Dataset.from_pandas(frequencies)
drug_rating_dataset = Dataset.from_pandas(avg_rating_per_drug)
print(freq_dataset)
print(drug_rating_dataset)

reset the output format of `drug_dataset` from `"pandas"` to `"arrow"`

In [ ]:
drug_dataset.reset_format()

## Creating a validation set

Datasets provides a `Dataset.train_test_split()` function that is based on the famous functionality from `scikit-learn`.

In [ ]:
drug_dataset_clean = drug_dataset["train"].train_test_split(train_size=0.8, seed=42)
# Rename the default "test" split to "validation"
drug_dataset_clean["validation"] = drug_dataset_clean.pop("test")
# Add the "test" set to our `DatasetDict`
drug_dataset_clean["test"] = drug_dataset["test"]
drug_dataset_clean

# Saving/Loading a Dataset to/from disk

- Datasets will cache every downloaded dataset and the operations performed on it
- Still, there are times when you’ll want to save a dataset to disk (e.g., in case the cache gets deleted).

`Dataset.save_to_disk()` --> saved as `Arrow`

`Dataset.to_csv()` --> saved as `CSV`

`Dataset.to_json()` --> saved as `JSON`


### Arrow format

In [ ]:
save_dir = "data/drug-reviews"
drug_dataset_clean.save_to_disk(save_dir)

In [ ]:
!tree {save_dir}

Once the dataset is saved, we can load it by using the `load_from_disk()` function as follows

In [ ]:
from datasets import load_from_disk

drug_dataset_reloaded = load_from_disk(save_dir)
drug_dataset_reloaded

### Json format

In [ ]:
for split, dataset in drug_dataset_clean.items():
    dataset.to_json(f"data/drug-reviews-{split}.jsonl")

In [ ]:
!head -n 1 data/drug-reviews-train.jsonl

In [ ]:
data_files = {
    "train": "data/drug-reviews-train.jsonl",
    "validation": "data/drug-reviews-validation.jsonl",
    "test": "data/drug-reviews-test.jsonl",
}
drug_dataset_reloaded = load_dataset("json", data_files=data_files)
drug_dataset_reloaded

# TODOs
- Use the techniques from [Chapter 3](https://huggingface.co/course/chapter3) to train a classifier that can predict the **patient condition** based on the **drug review**.
    - Use `Trainer` API
    - The challenge, not addressed yet, is to prepare labels for the trainer. (keep track of label values and names)
- Use the `summarization` pipeline from [Chapter 1](https://huggingface.co/course/chapter1) to generate summaries of the **reviews**.
    - This should as efficient/batched as possible
    - Bonus: make it even parallel

# Handling big data

- Datasets has been designed to handle big data.
- It frees you **from** memory management problems by treating datasets as **memory-mapped** files
- and **from** hard drive limits by streaming the entries in a corpus.

In [ ]:
from huggingface_hub import login
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN') # Retrieve the token from secrets
if HF_TOKEN:
    login(HF_TOKEN)
    print("Successfully logged in to Hugging Face!")
else:
    print("HF_TOKEN not found in Colab secrets.")

## The [Pile](https://pile.eleuther.ai/)


In [ ]:
from datasets import load_dataset, DownloadConfig

# # this code doesn't work as the-eye.eu is down right now!!!
# !pip install zstandard
# base_url = "https://the-eye.eu/public/AI/pile_preliminary_components"
# data_files = f"{base_url}/PUBMED_title_abstracts_2019_baseline.jsonl.zst"
# pubmed_dataset = load_dataset("json", data_files=data_files, split="train")
# pubmed_dataset

In [ ]:
base_url = "https://huggingface.co/datasets"
repo_url = "EleutherAI/pile/resolve/refs%2Fconvert%2Fparquet"
subset = "pubmed/partial/train"
data_files = [f"{base_url}/{repo_url}/{subset}/000{i}.parquet" for i in range(10)]

pubmed_dataset = load_dataset(
    "parquet",
    data_files=data_files,
    split="train",
    download_config = DownloadConfig(delete_extracted=True)
)

In [ ]:
pubmed_dataset

In [ ]:
pubmed_dataset[0]

In [ ]:
# !pip install psutil
import psutil
# Process.memory_info is expressed in bytes, so convert to megabytes
print(f"RAM used: {psutil.Process().memory_info().rss / (1024 * 1024):.2f} MB")

- Here the `rss` attribute refers to the resident set size, which is the fraction of memory that a process occupies in RAM.
- This measurement also includes the memory used by the Python interpreter and the libraries we’ve loaded
- So the actual amount of memory used to load the dataset is a bit smaller.
- For comparison, let’s see how large the dataset is on disk, using the `dataset_size` attribute

In [ ]:
print(f"Dataset size in bytes: {pubmed_dataset.dataset_size}")
size_gb = pubmed_dataset.dataset_size / (1024**3)
print(f"Dataset size (cache file) : {size_gb:.2f} GB")

how does 🤗 Datasets solve this memory management problem?
- 🤗 Datasets treats each dataset as a **[memory-mapped file](https://en.wikipedia.org/wiki/Memory-mapped_file)**, which provides a mapping between RAM and filesystem storage that allows the library to access and operate on elements of the dataset without needing to fully load it into memory.

- Memory-mapped files can also be **shared** across multiple processes, which enables methods like `Dataset.map()` to be parallelized without needing to move or copy the dataset.
- Under the hood, these capabilities are all realized by the **[Apache Arrow](https://arrow.apache.org/)** memory format and [`pyarrow`](https://arrow.apache.org/docs/python/index.html) library, which make the data loading and processing lightning fast.
- For more details about Apache Arrow and comparisons to Pandas, check out [Dejan Simic’s blog post](https://towardsdatascience.com/apache-arrow-read-dataframe-with-zero-memory-69634092b1a).
- To see this in action, let’s run a little speed test by iterating over all the elements in the PubMed Abstracts dataset:

In [ ]:
import timeit

code_snippet = """batch_size = 1000

for idx in range(0, len(pubmed_dataset), batch_size):
    _ = pubmed_dataset[idx:idx + batch_size]
"""

time = timeit.timeit(stmt=code_snippet, number=1, globals=globals())
print(
    f"Iterated over {len(pubmed_dataset)} examples (about {size_gb:.1f} GB) in "
    f"{time:.1f}s, i.e. {size_gb/time:.3f} GB/s"
)

## Streaming datasets

- Sometimes we have to work with a dataset that is too large to even store on our laptop’s hard drive.
- To handle these cases, 🤗 Datasets provides a streaming feature that allows us to download and access elements on the fly, without needing to download the whole dataset

In [ ]:
pubmed_dataset_streamed = load_dataset(
    "parquet",
    data_files=data_files,
    split="train",
    download_config = DownloadConfig(delete_extracted=True),
    streaming=True,
)

- Instead of the familiar `Dataset` that we’ve encountered elsewhere in this chapter, the object returned with `streaming=True` is an `IterableDataset`.
- As the name suggests, to access the elements of an IterableDataset we need to iterate over it.

In [ ]:
next(iter(pubmed_dataset_streamed))

- The elements from a streamed dataset can be processed on the fly using `IterableDataset.map()`, which is useful during training if you need to tokenize the inputs.
- The process is exactly the same as the one we used to tokenize our dataset before, with the only difference being that outputs is also an `Iterable`.
- To speed up tokenization with streaming you can pass `batched=True`, as we saw in the last section.
  - It will process the examples batch by batch
  - the default batch size is 1,000 and can be specified with the `batch_size` argument.

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
tokenized_dataset = pubmed_dataset_streamed.map(lambda x: tokenizer(x["text"]))
next(iter(tokenized_dataset))

- You can also shuffle a streamed dataset using `IterableDataset.shuffle()`, but unlike `Dataset.shuffle()` this only shuffles the elements in a predefined `buffer_size`
- Examples are randomly sampled from this buffer, and the selected examples are replaced by new ones from the stream, maintaining a constant buffer size.
- It also shuffles the order of data shards if the dataset is sharded into multiple files.

In [ ]:
shuffled_dataset = pubmed_dataset_streamed.shuffle(buffer_size=10_000, seed=42)
next(iter(shuffled_dataset))

You can also select elements from a streamed dataset using the `IterableDataset.take()` and `IterableDataset.skip()` functions, which act in a similar way to [`Dataset.select()`](https://huggingface.co/docs/datasets/v4.6.0/en/package_reference/main_classes#datasets.Dataset.select)

In [ ]:
dataset_head = pubmed_dataset_streamed.take(5)
list(dataset_head)

In [ ]:
# Skip the first 1,000 examples and include the rest in the training set
train_dataset = shuffled_dataset.skip(1000)
# Take the first 1,000 examples for the validation set
validation_dataset = shuffled_dataset.take(1000)

## Combining datasets

🤗 Datasets provides an `interleave_datasets()` function that converts a **list** of `IterableDataset` objects into a single `IterableDataset`, where the elements of the new dataset are obtained by **alternating** among the source examples.

In [ ]:
subset = "free_law/partial/train"
data_files = [f"{base_url}/{repo_url}/{subset}/000{i}.parquet" for i in range(10)]

law_dataset_streamed = load_dataset(
    "parquet",
    data_files=data_files,
    split="train",
    download_config = DownloadConfig(delete_extracted=True),
    streaming=True,
)

next(iter(law_dataset_streamed))

In [ ]:
from itertools import islice
from datasets import interleave_datasets

combined_dataset = interleave_datasets([pubmed_dataset_streamed, law_dataset_streamed])
tmp_sample = list(islice(combined_dataset, 2))

In [ ]:
print(tmp_sample[0]['meta'])
print(tmp_sample[1]['meta'])

# Creating your own dataset

- We’ll create a corpus of [GitHub issues](https://github.com/features/issues/) of 🤗 Datasets' repo
- Issues are commonly used to track bugs or features in GitHub repositories.
- This corpus could be used for various purposes, including:
  - Exploring how long it takes to close open issues or pull requests
  - Training a _multilabel classifier_ that can tag issues with metadata based on the issue’s description
    - (e.g., “bug,” “enhancement,” or “question”)
  - Creating a semantic search engine to find which issues match a user’s query
    - We’ll work on this in the next section

In [ ]:
# !pip install requests

In [ ]:
from google.colab import userdata
import requests
import time
import json
import math
import random
from pathlib import Path
import pandas as pd
from tqdm.notebook import tqdm
from datasets import load_dataset, Dataset, Value, List, Features

GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
headers = {"Authorization": f"token {GITHUB_TOKEN}"}

## Fetch data

In [ ]:
def fetch_issues(
    owner="huggingface",
    repo="datasets",
    num_issues=10_000,
    rate_limit=10_000,
    issues_path=Path("."),
):
    if not issues_path.is_dir():
        issues_path.mkdir(exist_ok=True)

    batch = []
    all_issues = []
    per_page = 100  # Number of issues to return per page
    num_pages = math.ceil(num_issues / per_page)
    base_url = "https://api.github.com/repos"

    for page in tqdm(range(num_pages)):
        # Query with state=all to get both open and closed issues
        query = f"issues?page={page}&per_page={per_page}&state=all"
        issues = requests.get(f"{base_url}/{owner}/{repo}/{query}", headers=headers)
        batch.extend(issues.json())

        if len(batch) > rate_limit and len(all_issues) < num_issues:
            all_issues.extend(batch)
            batch = []  # Flush batch for next time period
            print(f"Reached GitHub rate limit. Sleeping for one hour ...")
            time.sleep(60 * 60 + 1)

    all_issues.extend(batch)
    df = pd.DataFrame.from_records(all_issues)
    df.to_json(f"{issues_path}/{repo}-issues.jsonl", orient="records", lines=True)
    print(
        f"Downloaded all the issues for {repo}! Dataset stored at {issues_path}/{repo}-issues.jsonl"
    )

In [ ]:
# Depending on the internet connection, this can take several minutes to run...
fetch_issues()

## Infer schema and Load data as HF datasets

In [ ]:
def read_jsonl_random_to_df(path: str, n: int = 200, seed: int = 42) -> pd.DataFrame:
    random.seed(seed)
    reservoir = []

    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            obj = json.loads(line)

            if i < n:
                reservoir.append(obj)
            else:
                j = random.randint(0, i)
                if j < n:
                    reservoir[j] = obj

    return pd.DataFrame(reservoir)

sample_df = read_jsonl_random_to_df("datasets-issues.jsonl", n=500)
sample_ds = Dataset.from_pandas(sample_df, preserve_index=False)

print(sample_ds.features)

In [ ]:
patched = {}
def patch_recur(dtype_dict: dict | Features) -> dict:
  patched = {}
  for k, feat in dtype_dict.items():
    if isinstance(feat, Value):
      dtype = feat.dtype
      if dtype == "null":
        patched[k] = Value("string")
      else:
        patched[k] = feat
    elif isinstance(feat, List):
      inner_feat = feat.feature
      if isinstance(inner_feat, Value):
        patched[k] = List(Value("string"))
      elif isinstance(inner_feat, dict):
        patched[k] = List(patch_recur(feat.feature))
      else: raise ValueError(f"Unknown feature type inside a List: {inner_feat}")
    elif isinstance(feat, dict):
      patched[k] = patch_recur(feat)
    else: raise ValueError(f"Unknown feature type: {feat}")
  return patched

patched_features = patch_recur(sample_ds.features)
print(patched_features)

In [ ]:
issues_dataset = load_dataset(
    "json",
    data_files="datasets-issues.jsonl",
    split="train",
    features=Features(patched_features),
)
issues_dataset

## Cleaning up the data

GitHub’s REST API v3 considers every pull request an issue, but not every issue is a pull request. For this reason, “Issues” endpoints may return both issues and pull requests in the response. You can identify pull requests by the `pull_request` key. Be aware that the `id` of a pull request returned from “Issues” endpoints will be an issue id.

In [ ]:
sample = issues_dataset.shuffle(seed=666).select(range(3))
# Print out the URL and pull request entries
for url, pr in zip(sample["html_url"], sample["pull_request"]):
    print(f">> URL: {url}")
    print(f">> Pull request: {pr}\n")

## EDA examples

In [ ]:
issues_dataset = issues_dataset.map(
    lambda x: {"is_pull_request": [False if y is None else True for y in x["pull_request"]]},
    batched=True,
)

### Issue avg close time

In [ ]:
close_time_df = issues_dataset.filter(
    lambda x: [(y == False) and (z is not None) for (y,z) in zip(x["is_pull_request"], x["closed_at"])],
    batched=True,
).select_columns(["closed_at", "created_at"]).to_pandas()

close_time_df["close_time"] = pd.to_datetime(close_time_df["closed_at"]) - pd.to_datetime(close_time_df["created_at"])
close_time_df["close_time"].mean()

### PR avg close time

In [ ]:
close_time_df = issues_dataset.filter(
    lambda x: [(y == True) and (z is not None) for (y,z) in zip(x["is_pull_request"], x["closed_at"])],
    batched=True,
).select_columns(["closed_at", "created_at"]).to_pandas()

close_time_df["close_time"] = pd.to_datetime(close_time_df["closed_at"]) - pd.to_datetime(close_time_df["created_at"])
close_time_df["close_time"].mean()

## Augmenting the dataset

- The comments associated with an issue or pull request provide a rich source of information
- Especially if we’re interested in building a search engine to answer user queries about the library.
- The GitHub REST API provides a [`Comments` endpoint](https://docs.github.com/en/rest/reference/issues#list-issue-comments) that returns all the comments associated with an issue number

In [ ]:
import requests
import time
from datetime import datetime

BASE_URL = "https://api.github.com"
REPO = "huggingface/datasets"

def check_rate_limit():
    response = requests.get(f"{BASE_URL}/rate_limit", headers=headers)
    data = response.json()["rate"]

    limit = data["limit"]
    remaining = data["remaining"]
    reset_timestamp = data["reset"]

    reset_time = datetime.utcfromtimestamp(reset_timestamp)

    print(f"Rate limit: {remaining}/{limit} remaining")
    print(f"Resets at (UTC): {reset_time}")

    return limit, remaining, reset_timestamp

# Optional: Check initial rate limit
check_rate_limit()

In [ ]:
def adaptive_sleep(remaining, reset_timestamp):
    """
    Adjust sleep time dynamically depending on how many
    requests remain before reset.
    """
    if remaining <= 0:
        sleep_time = max(reset_timestamp - time.time(), 0) + 1
        print(f"Rate limit exceeded. Sleeping {sleep_time:.2f} seconds...")
        time.sleep(sleep_time)
    elif remaining < 50:
        # If getting close to limit, slow down
        print("Approaching rate limit. Slowing down...")
        time.sleep(2)
    else:
        # Normal slight delay to avoid burst hitting
        time.sleep(0.2)


def get_comments(issue_number):
    url = f"{BASE_URL}/repos/{REPO}/issues/{issue_number}/comments"

    response = requests.get(url, headers=headers)

    # Extract rate limit headers
    limit = int(response.headers.get("X-RateLimit-Limit", 0))
    remaining = int(response.headers.get("X-RateLimit-Remaining", 0))
    reset_timestamp = int(response.headers.get("X-RateLimit-Reset", 0))

    # print(f"Issue {issue_number} | Remaining: {remaining}/{limit}")

    if response.status_code == 403 and remaining == 0:
        adaptive_sleep(remaining, reset_timestamp)
        return get_comments(issue_number)  # retry after sleep

    if response.ok:
        adaptive_sleep(remaining, reset_timestamp)
        return [r["body"] for r in response.json()]
    else:
        print(f"Error fetching issue {issue_number}: {response.status_code}")
        return []

In [ ]:
issues_with_comments_dataset = issues_dataset.map(
    lambda x: {"comments": get_comments(x["number"])}
)

## Push dataset to HF hub

In [ ]:
from huggingface_hub import login
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN') # Retrieve the token from secrets
if HF_TOKEN:
    login(HF_TOKEN)
    print("Successfully logged in to Hugging Face!")
else:
    print("HF_TOKEN not found in Colab secrets.")

In [ ]:
issues_with_comments_dataset.push_to_hub("github-issues")

In [ ]:
remote_dataset = load_dataset("amin-oj/github-issues", split="train")
remote_dataset

## Create dataset card on HF hub

`TODO`
- Use dataset card creation [guide](https://github.com/huggingface/datasets/blob/main/templates/README_guide.md) from the official HF repo
  - This works as both a template and a guide doc
- To create tags to be added to the readme file use this [online tagging app](https://huggingface.co/spaces/huggingface/datasets-tagging)

- this can be a minimal example tagging for our dataset:
```yaml
annotations_creators:
- found
language:
- en
language_creators:
- found
license:
- apache-2.0
multilinguality:
- monolingual
pretty_name: HF datasets GitHub issues
size_categories:
- 1K<n<10K
source_datasets:
- original
tags:
- github
- issue-query
- experimental
task_categories:
- sentence-similarity
- text-classification
- text-retrieval
task_ids:
- multi-class-classification
- multi-label-classification
- document-retrieval
```